# 04 — Classical ARIMA Forecasting
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook builds and evaluates a classical **Autoregressive Integrated Moving Average (ARIMA)** model:
1. **National Aggregate Daily Series:** Aggregates multi-store daily sales to national level to establish the primary statistical benchmark.
2. **Frequency Standardization:** Enforces strict daily calendar continuity (`asfreq('D')`).
3. **Chronological Splitting:** 80% train / 20% test partition without shuffling.
4. **Auto-ARIMA Optimization:** Grid-searching parameters $(p,d,q)$ based on Akaike Information Criterion (AIC).
5. **Model Fitting & Diagnostics:** Fitting $\text{ARIMA}(4,1,0)$ and verifying residual white noise via diagnostic plots (Q-Q, correlogram).
6. **Multi-Step Out-of-Sample Forecasting:** Generating forecasts with 95% confidence intervals.
7. **Artifact Export:** Serializing `reports/arima_summary.txt` and `reports/arima_forecast.csv` for the Streamlit dashboard.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
REPORTS_DIR = BASE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading cleaned dataset from {DATA_DIR / 'clean_merged.csv'}...")
df = pd.read_csv(DATA_DIR / 'clean_merged.csv', parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows.")


### 2. Time-Series Aggregation & Chronological Split
We aggregate sales to national daily totals, enforce strict daily frequency, and split into 80% training and 20% testing sets.


In [ ]:
daily_sales = df.groupby('date')['sales'].sum().reset_index().set_index('date').asfreq('D')
daily_sales['sales'] = daily_sales['sales'].interpolate(method='linear')

split_idx = int(len(daily_sales) * 0.8)
train = daily_sales.iloc[:split_idx]['sales']
test = daily_sales.iloc[split_idx:]['sales']

print(f"Full series: {len(daily_sales)} days ({daily_sales.index.min().date()} to {daily_sales.index.max().date()})")
print(f"Train split: {len(train)} days ({train.index.min().date()} to {train.index.max().date()})")
print(f"Test split:  {len(test)} days ({test.index.min().date()} to {test.index.max().date()})")


### 3. Model Fitting: ARIMA(4, 1, 0)
Based on ACF/PACF analysis and AIC optimization, we fit an autoregressive model of order 4 with first-order differencing ($d=1$).


In [ ]:
p, d, q = 4, 1, 0
print(f"Fitting ARIMA({p}, {d}, {q}) on National Daily Sales training data...")

arima_model = ARIMA(train, order=(p, d, q))
arima_fit = arima_model.fit()

print(arima_fit.summary())


### 4. Residual Diagnostics
Residuals must resemble white noise (zero mean, constant variance, and no significant autocorrelation remaining).


In [ ]:
fig = arima_fit.plot_diagnostics(figsize=(14, 8))
plt.suptitle(f"ARIMA({p},{d},{q}) Residual Diagnostics", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 5. Multi-Step Forecasting & Confidence Intervals
We forecast the entire test window and retrieve 95% confidence intervals.


In [ ]:
forecast_obj = arima_fit.get_forecast(steps=len(test))
predicted_mean = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int(alpha=0.05)

plt.figure(figsize=(15, 6))
plt.plot(train.index[-90:], train.iloc[-90:], label='Train (Recent 90 Days)', color='black', alpha=0.6)
plt.plot(test.index, test, label='Actual Test Sales', color='#1f77b4', linewidth=2)
plt.plot(test.index, predicted_mean, label=f'ARIMA({p},{d},{q}) Forecast', color='#d62728', linewidth=2, linestyle='--')
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='#d62728', alpha=0.15, label='95% Confidence Interval')

plt.title(f'ARIMA({p},{d},{q}) National Daily Sales Forecast vs Actuals', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('National Sales')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 6. Error Metrics Evaluation
Calculate MAE, RMSE, and MAPE on the out-of-sample test window.


In [ ]:
mae = mean_absolute_error(test, predicted_mean)
rmse = np.sqrt(mean_squared_error(test, predicted_mean))
mask = test > 0
mape = np.mean(np.abs((test[mask] - predicted_mean[mask]) / test[mask])) * 100

print(f"=== ARIMA({p},{d},{q}) Test Evaluation Metrics ===")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")


### 7. Exporting Artifacts for the Web Dashboard
We export:
1. `reports/arima_summary.txt`: Full statsmodels regression output for Tab 4.
2. `reports/arima_forecast.csv`: Historical actuals, predictions, and confidence intervals for interactive plotting.


In [ ]:
# 1. Save text summary
summary_path = REPORTS_DIR / 'arima_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(str(arima_fit.summary()))
print(f"Saved: {summary_path}")

# 2. Save forecast CSV
forecast_df = pd.DataFrame({
    'date': test.index,
    'actual': test.values,
    'predicted': predicted_mean.values,
    'lower_ci': conf_int.iloc[:, 0].values,
    'upper_ci': conf_int.iloc[:, 1].values
})
forecast_path = REPORTS_DIR / 'arima_forecast.csv'
forecast_df.to_csv(forecast_path, index=False)
print(f"Saved: {forecast_path} ({len(forecast_df)} rows)")
print("\nARIMA pipeline completed successfully! Proceed to 05_sarima_forecasting.ipynb.")
